### I. Mathematical Foundation: The Return Transformation

We do not feed raw price data $P$ into PCA or ICA. Prices are non-stationary, meaning their statistical properties (mean, variance) change over time. Unsupervised learning algorithms require **Stationarity** to find stable latent factors.

We transform prices into **Log-Returns** $r_t$. This transformation serves two purposes:

1. **Normalization:** It converts absolute currency changes into percentage-like changes, allowing us to compare a high-priced stock with a low-priced stock.
2. **Additivity:** Log-returns are time-additive, which is mathematically superior for long-term volatility modeling.

The formula for the log-return at time $t$ is:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

Before we can orchestrate the main entry point, we must formalize the **Data Alignment Layer**. In a high-dimensional financial engine, temporal synchronization is the "puzzle piece" that converts irregular market events into a dense matrix suitable for linear algebra.
___

### II. Mathematical Foundation: Temporal Alignment and Forward-Filling

When combining data from the Philippines (PSE), New York (NYSE), and Hong Kong (HKEX), the union of their timestamps $T = T_{PSE} \cup T_{NYSE} \cup T_{HKEX}$ creates a sparse index. Many indices will contain $NaN$ (Not a Number) values because one market was closed while another was open.

In Unsupervised Learning, the covariance matrix $\Sigma$ used in PCA cannot be computed if the input matrix $X$ contains missing values. We apply **Last Observation Carried Forward (LOCF)**.

Mathematically, if $P_{i,t}$ is the price of asset $i$ at time $t$, and $t$ is a holiday for that asset's exchange:


$$P_{i,t} = P_{i,t-k}$$


where $k$ is the smallest integer such that $P_{i,t-k}$ exists.

This assumes that the "fair value" of the asset remains unchanged until the next trading session. We follow this with a truncation step to remove the "all-NaN" rows at the beginning of the series where assets may not have started trading yet.

---
### III. Mathematical Foundation: Matrix Sparsity and Rank Deficiency

The goal of your ingestion is to produce a Feature Matrix $X \in \mathbb{R}^{T \times N}$. If one ticker fails, your matrix becomes $X \in \mathbb{R}^{T \times (N-1)}$. In an unsupervised learning context, specifically Principal Component Analysis (PCA), if the ingestion returns an empty dataset ($T=0$), the covariance matrix $\Sigma$ cannot be computed:

$$\Sigma = \frac{1}{T-1} \sum_{t=1}^{T} (x_t - \bar{x})(x_t - \bar{x})^T$$

If $\Sigma$ is undefined, the eigenvector decomposition $\Sigma \mathbf{v} = \lambda \mathbf{v}$ fails, halting the entire pipeline. We must ensure $X$ is a **Full Rank** matrix.